# AI-2027 — مختبر بروتوكول الإبهار البرمجي بايثون 2027 (Colab · $0)

تطبيق عملي لبنود البروتوكول الـ10 بتكلفة **$0**:
- `uv` + `ruff` + `Pydantic V2` + `asyncio.TaskGroup`
- `structlog` + `OpenTelemetry` تتبع حيّ
- توجيه **Free-First** (`Ollama` / `Gemini` / `Groq` / `OpenRouter` + احتياطي حتمي)
- تدقيق امتثال مستودع `ai-2027` + قياس الأداء قبل التحسين

In [ ]:
import os
import pathlib
import shutil
import subprocess
import sys

print(f"Python runtime: {sys.version.split()[0]}")
repo_dir = pathlib.Path("/content/ai-2027")
if pathlib.Path("python-2027/pyproject.toml").is_file():
    pkg_dir = pathlib.Path("python-2027").resolve()
elif pathlib.Path("../pyproject.toml").is_file():
    pkg_dir = pathlib.Path("..").resolve()
elif pathlib.Path("pyproject.toml").is_file():
    pkg_dir = pathlib.Path(".").resolve()
else:
    if not repo_dir.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/sayedelazameydesign-crypto/ai-2027.git", str(repo_dir)])
    pkg_dir = repo_dir / "python-2027"
os.chdir(pkg_dir)
if shutil.which("uv") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "uv"])
subprocess.check_call(["uv", "pip", "install", "--system", "-q", "-e", "."])
print(f"Installed ai2027-protocol from {pkg_dir}")

In [ ]:
import os

SECRET_NAMES = ("GEMINI_API_KEY", "NVIDIA_API_KEY", "GROQ_API_KEY", "OPENROUTER_API_KEY")
try:
    from google.colab import userdata  # type: ignore[import-not-found]
    for name in SECRET_NAMES:
        try:
            val = userdata.get(name)
            if val and not os.environ.get(name):
                os.environ[name] = val
        except Exception:
            pass
except ImportError:
    print("Running outside Google Colab; using environment variables or deterministic $0 fallback.")

status = {name: ("configured" if os.environ.get(name) else "absent (fallback active)") for name in SECRET_NAMES}
print("Provider keys status (names only):", status)

In [ ]:
import asyncio
import concurrent.futures
from ai2027_protocol import FreeFirstRouter, ObservabilityRuntime, PromptRequest

def run_coro(coro):
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(asyncio.run, coro).result()

async def run_colab_demo() -> None:
    runtime = ObservabilityRuntime(json_logs=True)
    router = FreeFirstRouter(telemetry=runtime)
    response = await router.route(PromptRequest(prompt="Explain why Evidence-First AI agents beat unverified claims."))
    print("Selected provider:", response.provider.value, "| simulated:", response.simulated, "| cost:$", response.cost_usd)
    print("Trace ID:", response.trace_id)
    print("Response:", response.text)
    batch = await router.route_batch([PromptRequest(prompt=f"TaskGroup task {i}") for i in range(4)], max_concurrency=4)
    print(f"TaskGroup concurrent batch completed: {len(batch)} items, OTel spans={len(runtime.exported_spans())}")
    runtime.shutdown()

run_coro(run_colab_demo())

In [ ]:
import pathlib
from ai2027_protocol import ProtocolAuditor, format_audit_markdown, run_benchmark_suite

async def run_measurements() -> None:
    bench = await run_benchmark_suite(iterations=12)
    for comp in bench.comparisons:
        print(f"{comp.baseline.name}: {comp.baseline.p50_ms}ms -> {comp.optimized.name}: {comp.optimized.p50_ms}ms ({comp.speedup_x}x)")
    root = pathlib.Path.cwd().parent
    auditor = ProtocolAuditor(workspace_root=root)
    targets = [p for p in ("python-2027", "connectors-hub", "projects/agi-system", "projects/1pro") if (root / p).is_dir()]
    report = await auditor.audit_projects(targets or ["."])
    print(format_audit_markdown(report))

run_coro(run_measurements())